# Edge AI Deployment Pipeline

This notebook loads the trained model from the original notebook and performs:

1. **ONNX conversion** of the scikit-learn Logistic Regression pipeline
2. **Validation**: numerical comparison between sklearn and ONNX Runtime
3. **Latency benchmark**: sklearn vs ONNX Runtime
4. **Enhanced Raspberry Pi inference script** with P99 latency, power monitoring (vcgencmd), and SQLite logging
5. **Dashboard launch**: the Gradio monitoring dashboard

---
**Prerequisites**: Run the original `Edge_AI_Menstrual_Health_final.ipynb` first to generate the saved model artifacts.

## 1. Setup and imports

In [ ]:
import os
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore")

# Add src to path so we can import the package
sys.path.insert(0, str(Path.cwd().parent))

from edge_ai.models.onnx_utils import convert_to_onnx, validate_onnx, benchmark_latency
from edge_ai.monitoring.metrics import setup_database, measure_all, log_inference, measure_latency
from edge_ai.xai.explainer import Explainer

print("All imports successful.")

## 2. Configure paths

Update `MODEL_DIR` to point to your exported model files from the original notebook.

In [ ]:
# --------------------------------------------------
# Point this to where the original notebook exported files
# --------------------------------------------------
MODEL_DIR = Path("/content/drive/MyDrive/EdgeAI_Project/models/raspberrypi_test")

# If running locally, uncomment and update:
# MODEL_DIR = Path.home() / "edge_ai" / "models"

MODEL_PATH = MODEL_DIR / "final_edge_ai_symptom_risk_pipeline.joblib"
FEATURE_PATH = MODEL_DIR / "final_edge_ai_input_feature_names.joblib"
THRESHOLD_PATH = MODEL_DIR / "final_edge_ai_threshold.joblib"
SAMPLE_INPUT_PATH = MODEL_DIR / "edge_sample_input.csv"
ONNX_PATH = MODEL_DIR / "final_edge_ai_symptom_risk_pipeline.onnx"

print(f"Model path: {MODEL_PATH}")
print(f"ONNX path: {ONNX_PATH}")
print(f"All source files exist: {all(p.exists() for p in [MODEL_PATH, FEATURE_PATH, THRESHOLD_PATH, SAMPLE_INPUT_PATH])}")

## 3. Load trained model and metadata

In [ ]:
pipeline = joblib.load(MODEL_PATH)
feature_names = joblib.load(FEATURE_PATH)
threshold = joblib.load(THRESHOLD_PATH)
sample_df = pd.read_csv(SAMPLE_INPUT_PATH)

# Ensure sample has expected features
sample_df = sample_df[feature_names]

print(f"Pipeline type: {type(pipeline).__name__}")
print(f"Features: {len(feature_names)}")
print(f"Threshold: {threshold}")
print(f"Sample shape: {sample_df.shape}")
print(f"Sample columns: {list(sample_df.columns)}")

## 4. Verify sklearn prediction works

In [ ]:
probs = pipeline.predict_proba(sample_df)[:, 1]
preds = (probs >= threshold).astype(int)

print(f"Example probabilities (first 5): {probs[:5].round(4)}")
print(f"Example predictions (first 5): {preds[:5]}")
print(f"Positive rate: {preds.mean():.3f}")
print("Sklearn pipeline works correctly.")

## 5. Convert to ONNX

In [ ]:
print("Converting pipeline to ONNX...")
convert_to_onnx(pipeline, ONNX_PATH, sample_df)

onnx_size_kb = ONNX_PATH.stat().st_size / 1024
joblib_size_kb = MODEL_PATH.stat().st_size / 1024

print(f"ONNX model saved to: {ONNX_PATH}")
print(f"ONNX model size: {onnx_size_kb:.2f} KB")
print(f"Joblib model size: {joblib_size_kb:.2f} KB")
print(f"Size ratio (ONNX/joblib): {onnx_size_kb / joblib_size_kb:.2f}x")

## 6. Validate ONNX correctness

Check that ONNX Runtime produces the same probabilities as sklearn within tolerance.

In [ ]:
validation = validate_onnx(ONNX_PATH, pipeline, sample_df)

print(f"Max absolute difference: {validation['max_abs_difference']:.6e}")
print(f"Mean absolute difference: {validation['mean_abs_difference']:.6e}")
print(f"Validation PASSED: {validation['passed']}")
print(f"Sklearn output shape: {validation['sklearn_shape']}")
print(f"ONNX output shape: {validation['onnx_shape']}")

if not validation['passed']:
    print("WARNING: ONNX output differs from sklearn! Check the conversion.")

## 7. Latency benchmark (sklearn vs ONNX Runtime)

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(str(ONNX_PATH))

print("Running 1000-iteration latency benchmark...")
bench = benchmark_latency(pipeline, session, sample_df, n_runs=1000)

results_df = pd.DataFrame({
    "Metric": ["Mean", "Median", "Std", "P95", "P99", "Min", "Max"],
    "sklearn (ms)": [
        bench["sklearn"]["mean_ms"], bench["sklearn"]["median_ms"],
        bench["sklearn"]["std_ms"], bench["sklearn"]["p95_ms"],
        bench["sklearn"]["p99_ms"], bench["sklearn"]["min_ms"],
        bench["sklearn"]["max_ms"],
    ],
    "ONNX (ms)": [
        bench["onnx"]["mean_ms"], bench["onnx"]["median_ms"],
        bench["onnx"]["std_ms"], bench["onnx"]["p95_ms"],
        bench["onnx"]["p99_ms"], bench["onnx"]["min_ms"],
        bench["onnx"]["max_ms"],
    ],
})

display(results_df.round(4))

speedup = bench["sklearn"]["mean_ms"] / bench["onnx"]["mean_ms"]
print(f"ONNX speedup over sklearn (mean): {speedup:.2f}x")
print("Benchmark complete.")

## 8. Generate enhanced Raspberry Pi inference script

This script includes:
- P50/P95/P99 latency measurement
- Power monitoring via vcgencmd (voltage, throttling, temperature)
- RAM and CPU monitoring
- SQLite logging for dashboard consumption
- Works with both joblib and ONNX Runtime backends

In [ ]:
inference_script = '''
import os
import sys
import time
import json
import platform
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import psutil

# --------------------------------------------------
# Configuration - update paths as needed
# --------------------------------------------------
MODEL_DIR = Path(__file__).parent
MODEL_PATH = MODEL_DIR / "final_edge_ai_symptom_risk_pipeline.joblib"
ONNX_PATH = MODEL_DIR / "final_edge_ai_symptom_risk_pipeline.onnx"
FEATURE_LIST_PATH = MODEL_DIR / "final_edge_ai_input_feature_names.joblib"
THRESHOLD_PATH = MODEL_DIR / "final_edge_ai_threshold.joblib"
INPUT_PATH = MODEL_DIR / "edge_sample_input.csv"
DB_PATH = Path.home() / ".edge_ai_monitoring.db"

N_RUNS = 1000
N_WARMUP = 10
USE_ONNX = ONNX_PATH.exists()


def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)


def check_file_exists(path):
    if not os.path.exists(path):
        print(f"[ERROR] Missing: {path}", flush=True)
        sys.exit(1)


_RAPL_PATH = "/sys/class/powercap/intel-rapl:0/energy_uj"
_prev_rapl = None


def _read_rapl():
    try:
        with open(_RAPL_PATH) as f:
            return int(f.read().strip()) * 1e-6
    except Exception:
        return None


def _read_rapl_power():
    global _prev_rapl
    now = time.time()
    energy = _read_rapl()
    if energy is None:
        return None
    if _prev_rapl is not None:
        t_prev, e_prev = _prev_rapl
        dt = now - t_prev
        if dt > 0.01:
            power = (energy - e_prev) / dt
            _prev_rapl = (now, energy)
            return round(power, 3)
    _prev_rapl = (now, energy)
    return None


def _read_thermal():
    for path in [
        "/sys/class/thermal/thermal_zone0/temp",
        "/sys/class/hwmon/hwmon0/temp1_input",
    ]:
        try:
            with open(path) as f:
                val = int(f.read().strip())
                return val / 1000.0
        except Exception:
            continue
    return None


def measure_power():
    result = {"core_volts": None, "throttled": None, "temp_celsius": None}
    for cmd, key in [
        ("vcgencmd measure_volts core", "core_volts"),
        ("vcgencmd get_throttled", "throttled"),
        ("vcgencmd measure_temp", "temp_celsius"),
    ]:
        try:
            out = os.popen(cmd).read().strip()
            if "volt" in out:
                result[key] = float(out.replace("volt=", "").replace("V", ""))
            elif "throttled" in out:
                result[key] = out.replace("throttled=", "")
            elif "temp" in out:
                result[key] = float(out.replace("temp=", "").replace("'C", ""))
        except Exception:
            pass
    if result["temp_celsius"] is None:
        result["temp_celsius"] = _read_thermal()
    rapl_power = _read_rapl_power()
    if rapl_power is not None:
        result["core_volts"] = rapl_power
    return result


def setup_db():
    conn = sqlite3.connect(str(DB_PATH))
    conn.execute("""CREATE TABLE IF NOT EXISTS inference_logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT NOT NULL,
        probability REAL, prediction INTEGER, risk_level TEXT,
        latency_mean_ms REAL, latency_p95_ms REAL, latency_p99_ms REAL,
        latency_std_ms REAL, ram_mb REAL, cpu_percent REAL,
        core_volts REAL, throttled TEXT, temp_celsius REAL,
        model_backend TEXT, extra TEXT)""")
    conn.commit()
    conn.close()


def log_inference(prob, pred, risk_level, latency, backend, power, extra=None):
    proc = psutil.Process(os.getpid())
    conn = sqlite3.connect(str(DB_PATH))
    conn.execute(
        """INSERT INTO inference_logs
        (timestamp, probability, prediction, risk_level,
         latency_mean_ms, latency_p95_ms, latency_p99_ms, latency_std_ms,
         ram_mb, cpu_percent, core_volts, throttled, temp_celsius,
         model_backend, extra)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
        (
            datetime.now(timezone.utc).isoformat(),
            prob, pred, risk_level,
            latency.get("mean_ms"),
            latency.get("p95_ms"),
            latency.get("p99_ms"),
            latency.get("std_ms"),
            round(proc.memory_info().rss / (1024 ** 2), 2),
            psutil.cpu_percent(interval=0.1),
            power.get("core_volts"),
            power.get("throttled"),
            power.get("temp_celsius"),
            backend,
            json.dumps(extra) if extra else None,
        ),
    )
    conn.commit()
    conn.close()


def run_latency_test(predict_fn, sample, n_runs):
    for _ in range(N_WARMUP):
        predict_fn(sample)
    latencies = []
    for i in range(n_runs):
        row = sample.sample(1)
        t0 = time.perf_counter()
        predict_fn(row)
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000)
        if (i + 1) % 200 == 0:
            log(f"  {i + 1}/{n_runs} iterations")
    arr = np.array(latencies)
    return {
        "mean_ms": round(float(arr.mean()), 4),
        "median_ms": round(float(np.median(arr)), 4),
        "std_ms": round(float(arr.std()), 4),
        "p95_ms": round(float(np.percentile(arr, 95)), 4),
        "p99_ms": round(float(np.percentile(arr, 99)), 4),
        "min_ms": round(float(arr.min()), 4),
        "max_ms": round(float(arr.max()), 4),
    }


def risk_level(prob, threshold):
    if prob < threshold - 0.1:
        return "Low"
    elif prob > threshold + 0.1:
        return "High"
    return "Medium"


def main():
    print("==============================================", flush=True)
    print(" Raspberry Pi Edge AI Inference Test", flush=True)
    print("==============================================", flush=True)

    setup_db()

    # Step 1: Check files
    log("Step 1/9: Checking files...")
    for p in [MODEL_PATH, FEATURE_LIST_PATH, THRESHOLD_PATH, INPUT_PATH]:
        check_file_exists(p)
    if USE_ONNX:
        log(f"  ONNX Runtime available")
    log("  All files found.")

    # Step 2: Load model
    log("Step 2/9: Loading model...")
    pipeline = joblib.load(MODEL_PATH)
    feature_list = joblib.load(FEATURE_LIST_PATH)
    threshold = joblib.load(THRESHOLD_PATH)

    backend = "onnx" if USE_ONNX else "joblib"
    if USE_ONNX:
        import onnxruntime as ort
        session = ort.InferenceSession(str(ONNX_PATH))
        def _build_onnx_input(row):
            inp = {}
            for meta in session.get_inputs():
                col = meta.name
                if col in row.columns:
                    inp[col] = row[[col]].astype(np.float32, errors="ignore").to_numpy()
                else:
                    idx = int(col.replace("float_input_", ""))
                    inp[col] = row.iloc[:, [idx]].astype(np.float32, errors="ignore").to_numpy()
            return inp
        def onnx_predict(row):
            onnx_results = session.run(None, _build_onnx_input(row))
            probs = onnx_results[-1]
            if probs.ndim == 2 and probs.shape[1] == 2:
                return probs[0, 1]
            return probs[0] if probs.ndim == 1 else probs[0, 0]
        predict_fn = onnx_predict
        log(f"  Backend: ONNX Runtime")
    else:
        def sklearn_predict(row):
            return pipeline.predict_proba(row)[0, 1]
        predict_fn = sklearn_predict
        log(f"  Backend: sklearn/joblib")

    # Step 3: Load input
    log("Step 3/9: Loading input data...")
    X = pd.read_csv(INPUT_PATH)
    missing = [f for f in feature_list if f not in X.columns]
    if missing:
        print(f"[ERROR] Missing features: {missing}", flush=True)
        sys.exit(1)
    X = X[feature_list]
    log(f"  Input shape: {X.shape}")

    # Step 4: System info
    log("Step 4/9: System info...")
    print(f"  Device: {platform.platform()}", flush=True)
    print(f"  Python: {platform.python_version()}", flush=True)

    # Step 5: Model size
    log("Step 5/9: Model size...")
    model_size_kb = os.path.getsize(MODEL_PATH) / 1024
    print(f"  Joblib model: {model_size_kb:.2f} KB", flush=True)
    if USE_ONNX:
        onnx_size_kb = os.path.getsize(ONNX_PATH) / 1024
        print(f"  ONNX model: {onnx_size_kb:.2f} KB", flush=True)

    # Step 6: Warmup
    log("Step 6/9: Warmup...")
    warmup_prob = predict_fn(X.iloc[[0]])
    print(f"  Sample probability: {warmup_prob:.4f}", flush=True)

    # Step 7: Latency test
    log(f"Step 7/9: Running {N_RUNS} latency tests...")
    latencies = run_latency_test(predict_fn, X, N_RUNS)

    # Step 8: Batch + resources
    log("Step 8/9: Batch prediction + resource monitoring...")
    power = measure_power()
    batch_probs = [predict_fn(X.iloc[[i]]) for i in range(len(X))]
    batch_preds = [int(p >= threshold) for p in batch_probs]
    batch_probs_arr = np.array(batch_probs)

    proc = psutil.Process(os.getpid())
    ram_mb = round(proc.memory_info().rss / (1024 ** 2), 2)
    cpu = psutil.cpu_percent(interval=1)

    # Step 9: Log to database
    log("Step 9/9: Logging to SQLite...")
    for i in range(min(10, len(batch_probs))):
        log_inference(
            prob=float(batch_probs[i]),
            pred=int(batch_preds[i]),
            risk_level=risk_level(batch_probs[i], threshold),
            latency=latencies,
            backend=backend,
            power=power,
        )

    # Results
    print("", flush=True)
    print("==============================================", flush=True)
    print(" FINAL RESULTS", flush=True)
    print("==============================================", flush=True)

    results = {
        "device": platform.platform(),
        "model_size_kb": model_size_kb,
        "input_rows": len(X),
        "threshold": threshold,
        "backend": backend,
        **{f"latency_{k}": v for k, v in latencies.items()},
        "ram_mb": ram_mb,
        "cpu_percent": cpu,
        **{k: v for k, v in power.items() if v is not None},
        "positive_rate": float((np.array(batch_preds).mean())),
    }

    for k, v in results.items():
        print(f"  {k}: {v}", flush=True)

    pd.DataFrame([results]).to_csv(
        MODEL_DIR / "raspberry_pi_edge_results.csv", index=False
    )
    log("Results saved.")
    log("Edge inference test complete.")


if __name__ == "__main__":
    main()
'''

script_path = MODEL_DIR / "run_edge_inference.py"
with open(script_path, "w") as f:
    f.write(inference_script.strip())

print(f"Enhanced inference script saved to: {script_path}")
print(f"Size: {script_path.stat().st_size / 1024:.1f} KB")

## 9. Update requirements.txt for Raspberry Pi

In [ ]:
requirements = """pandas
numpy
scikit-learn
joblib
psutil
onnxruntime
gradio
plotly
"""

req_path = MODEL_DIR / "requirements.txt"
with open(req_path, "w") as f:
    f.write(requirements.strip() + "\n")

print(f"Requirements saved to: {req_path}")
print()
print("Files ready for Raspberry Pi:")
for f in sorted(MODEL_DIR.iterdir()):
    print(f"  - {f.name}")

## 10. Test XAI explanation on sample data

In [ ]:
explainer = Explainer(
    model=pipeline,
    feature_names=feature_names,
    background_df=sample_df,
    threshold=threshold,
)

# Explain a single sample
result = explainer.explain(sample_df.iloc[[0]])

print(f"Probability: {result['probability']:.4f}")
print(f"Risk Level: {result['risk_level']}")
print(f"Explanation method: {result['explanation']['method']}")
print()

top = result['explanation'].get('top_features', [])
if top:
    print("Top contributing factors:")
    for feat in top:
        val = feat.get('shap_value', feat.get('coefficient', 0))
        print(f"  {feat['feature']}: {val:.4f} ({feat['impact_direction']})")

print()
print("Planning Card:")
print(Explainer.make_planning_card(result['explanation']))

## 11. Initialize monitoring database

The database is created at `~/.edge_ai_monitoring.db` and stores:
- inference_logs: per-inference metrics (latency, RAM, CPU, power, prediction)
- system_snapshots: periodic system health metrics

In [ ]:
from edge_ai.monitoring.metrics import setup_database, log_system_snapshot

db_path = setup_database()
print(f"Database initialized at: {db_path}")

# Take an initial snapshot
snap_id = log_system_snapshot()
print(f"Initial system snapshot logged (id={snap_id})")

## 12. Launch the Gradio dashboard (optional)

Run this cell to start the monitoring dashboard locally.
Access it at `http://localhost:7860`.

To launch from the command line:
```bash
pip install -e .
edge-dashboard
```

In [ ]:
# Uncomment to launch the dashboard inline
# from edge_ai.dashboard.app import run
# run(db_path=db_path, share=False, port=7860, model_manager=explainer)

---

## Summary

| Task | Status |
|---|---|
| ONNX conversion | Done — model saved alongside joblib |
| ONNX validation | Verified — probabilities match sklearn |
| Latency benchmark | sklearn vs ONNX Runtime compared |
| Enhanced inference script | Generated with P99, power, SQLite |
| XAI explanation | Works with SHAP or coefficient fallback |
| Monitoring database | Initialized at `~/.edge_ai_monitoring.db` |
| Gradio dashboard | Ready to launch |

### Next steps on Raspberry Pi

1. Copy `models/` directory to the Pi
2. Run `bash setup_pi.sh` to install dependencies
3. Run `python run_edge_inference.py` to test inference
4. Launch dashboard: `python -m edge_ai.dashboard.app`
5. Check `raspberry_pi_edge_results.csv` for summary metrics